In [0]:
%sql
-- Databricks notebook source
-- MAGIC %md
-- MAGIC # Capa Gold - Agregaciones de Negocio
-- MAGIC **Objetivo:** Crear tablas finales listas para ser consumidas por el dashboard.

-- COMMAND ----------

-- 1. Tabla Resumen General (KPIs)
CREATE TABLE IF NOT EXISTS gold_market_summary (
    total_monedas INT, 
    capitalizacion_total_usd DOUBLE, 
    volumen_total_24h_usd DOUBLE, 
    variacion_promedio_24h DOUBLE, 
    ultima_actualizacion TIMESTAMP, 
    ingestion_date DATE
) USING DELTA;

-- 1.2. Insertar SOLO los datos de hoy (para no duplicar el histórico)
INSERT INTO gold_market_summary
SELECT 
    COUNT(id) AS total_monedas,
    ROUND(SUM(market_cap_usd), 2) AS capitalizacion_total_usd,
    ROUND(SUM(total_volume_usd), 2) AS volumen_total_24h_usd,
    ROUND(AVG(price_change_pct_24h), 2) AS variacion_promedio_24h,
    CURRENT_TIMESTAMP() AS ultima_actualizacion,
    CURRENT_DATE() AS ingestion_date
FROM silver_coingecko_market
WHERE ingestion_date = CURRENT_DATE(); -- Solo procesamos el día de hoy

-- COMMAND ----------

-- 2. Tabla Top 10 Monedas por Capitalización
CREATE TABLE IF NOT EXISTS gold_top_10_market_cap (
    market_cap_rank INT,
    name STRING,
    symbol STRING,
    current_price_usd DOUBLE,
    market_cap_usd DOUBLE,
    price_change_pct_24h DOUBLE,
    ingestion_date DATE
) USING DELTA;

INSERT INTO gold_top_10_market_cap
SELECT 
    market_cap_rank,
    name,
    symbol,
    current_price_usd,
    market_cap_usd,
    price_change_pct_24h,
    CURRENT_DATE() AS ingestion_date
FROM silver_coingecko_market
WHERE ingestion_date = CURRENT_DATE() -- Solo traemos los datos de hoy
ORDER BY market_cap_rank ASC
LIMIT 10;

-- COMMAND ----------

-- 3. Tabla Top 5 Ganadoras (Mayor suba en 24h)
CREATE TABLE IF NOT EXISTS gold_top_gainers (
    name STRING,
    symbol STRING,
    current_price_usd DOUBLE,
    price_change_pct_24h DOUBLE,
    ingestion_date DATE
) USING DELTA;

INSERT INTO gold_top_gainers
SELECT 
    name,
    symbol,
    current_price_usd,
    price_change_pct_24h,
    CURRENT_DATE() AS ingestion_date
FROM silver_coingecko_market
WHERE ingestion_date = CURRENT_DATE() 
  AND price_change_pct_24h IS NOT NULL
ORDER BY price_change_pct_24h DESC
LIMIT 5;

-- COMMAND ----------

-- 4. Tabla Top 5 Perdedoras (Mayor caída en 24h)
CREATE TABLE IF NOT EXISTS gold_top_losers (
    name STRING,
    symbol STRING,
    current_price_usd DOUBLE,
    price_change_pct_24h DOUBLE,
    ingestion_date DATE
) USING DELTA;

INSERT INTO gold_top_losers
SELECT 
    name,
    symbol,
    current_price_usd,
    price_change_pct_24h,
    CURRENT_DATE() AS ingestion_date
FROM silver_coingecko_market
WHERE ingestion_date = CURRENT_DATE() 
  AND price_change_pct_24h IS NOT NULL
ORDER BY price_change_pct_24h ASC
LIMIT 5;

-- COMMAND ----------

-- 5. LIMPIEZA: Borrar datos más viejos a 7 días en TODAS las tablas Gold
-- DELETE FROM bronze_coingecko_market WHERE CAST(ingestion_timestamp AS DATE) < CURRENT_DATE() - 7;
-- DELETE FROM silver_coingecko_market WHERE ingestion_date < CURRENT_DATE() - 7;
-- DELETE FROM gold_market_summary WHERE ingestion_date < CURRENT_DATE() - 7;
-- DELETE FROM gold_top_10_market_cap WHERE ingestion_date < CURRENT_DATE() - 7;
-- DELETE FROM gold_top_gainers WHERE ingestion_date < CURRENT_DATE() - 7;
-- DELETE FROM gold_top_losers WHERE ingestion_date < CURRENT_DATE() - 7;